# 🚗 Proyecto Integrador — Big Data & Machine Learning
## Especializaciones Implícitas de Vendedores y Predicción de Ventas de Alto Valor

| | |
|---|---|
| **Materia** | Prácticas y Herramientas de Big Data / Machine Learning |
| **Integrantes** | Jose Alexander Quishpe Reinoso · Julián Garofalo |
| **Dataset** | Car Sales Data (~2,500,000 registros) |
| **Carrera** | Ingeniería en Sistemas de Información |

---

## Índice

1. [Introducción](#1)
2. [Descripción del Dataset](#2)
3. [Hipótesis y Objetivos](#3)
4. [Análisis Exploratorio de Datos (EDA)](#4)
   - 4.1 [Estructura del Dataset](#41)
   - 4.2 [Calidad de Datos](#42)
   - 4.3 [Valores Faltantes](#43)
   - 4.4 [Outliers](#44)
   - 4.5 [Distribuciones](#45)
5. [Limpieza y Preprocesamiento](#5)
   - 5.1 [Eliminación de Duplicados](#51)
   - 5.2 [Tratamiento de Nulos](#52)
   - 5.3 [Conversión de Tipos](#53)
   - 5.4 [Ingeniería de Características](#54)
6. [Construcción de la Variable Objetivo](#6)
7. [Selección y Justificación de Variables](#7)
8. [División de Entrenamiento y Prueba](#8)
9. [Entrenamiento de Modelos](#9)
10. [Evaluación de Modelos](#10)
11. [Validación Cruzada](#11)
12. [Importancia de Variables](#12)
13. [Validación de Hipótesis](#13)
14. [Conclusiones](#14)


<a id='1'></a>
---
# 1. Introducción

El sector automotriz constituye uno de los mercados más dinámicos y económicamente relevantes a nivel global, caracterizado por transacciones de alto valor, una fuerte dependencia del factor humano en el proceso comercial y una creciente disponibilidad de datos transaccionales digitalizados. Esta combinación lo convierte en un escenario idóneo para la aplicación de técnicas de **Big Data** y **Machine Learning** orientadas a la generación de conocimiento accionable para la toma de decisiones gerenciales.

El presente proyecto integrador aborda un dataset de **aproximadamente 2,500,000 registros transaccionales** de ventas de vehículos, con el propósito de responder una pregunta de negocio concreta: *¿el desempeño de un vendedor —medido a través de su capacidad de concretar ventas de alto valor— constituye un patrón detectable y predecible mediante algoritmos de clasificación supervisada?*

Para responder esta pregunta de forma metodológicamente rigurosa, el proyecto sigue el flujo estándar de un proceso de Ciencia de Datos (similar a metodologías como **CRISP-DM**):

1. **Comprensión del negocio**: definición de la hipótesis y los objetivos (Sección 3).
2. **Comprensión y preparación de los datos**: exploración, auditoría de calidad y limpieza (Secciones 4 y 5).
3. **Modelado**: construcción de la variable objetivo, selección de variables y entrenamiento de tres algoritmos de complejidad creciente (Secciones 6 a 9).
4. **Evaluación**: comparación de métricas, validación cruzada y análisis de importancia de variables (Secciones 10 a 12).
5. **Conclusión**: validación formal de la hipótesis e implicancias para la gestión comercial (Secciones 13 y 14).

Esta estructura garantiza que cada decisión técnica —desde el tratamiento de un valor nulo hasta la elección del umbral de clasificación— esté respaldada por una justificación explícita, cumpliendo con los estándares esperados en un proyecto final de carácter universitario.


<a id='2'></a>
---
# 2. Descripción del Dataset

## 2.1. Variables Disponibles

El dataset **Car Sales Data** contiene registros transaccionales individuales de ventas de vehículos realizadas por una concesionaria automotriz. Cada fila representa una transacción única.

| Variable | Tipo | Descripción |
|---|---|---|
| `Date` | Fecha | Fecha exacta de la transacción |
| `Salesperson` | Categórica | Nombre del asesor comercial que gestionó la venta |
| `Customer Name` | Categórica | Nombre del comprador |
| `Car Make` | Categórica | Fabricante del vehículo (Toyota, Honda, Ford, Chevrolet, Nissan) |
| `Car Model` | Categórica | Modelo específico del automóvil |
| `Car Year` | Numérica Discreta | Año de fabricación del vehículo |
| `Sale Price` | Numérica Continua | Precio de venta final en USD |
| `Commission Rate` | Numérica Continua | Porcentaje de comisión asignado a la venta |
| `Commission Earned` | Numérica Continua | Monto total de comisión generado |

## 2.2. Origen y Extracción de los Datos

Se plantea una arquitectura de Big Data donde, en primer lugar, se verifica la conectividad con servicios externos (simulando el consumo de una API RESTful pública como parte del flujo de ingesta) y, a continuación, se realiza la lectura masiva del archivo CSV de ~2.5 millones de registros desde el almacenamiento de Google Colab.


In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
warnings.filterwarnings('ignore')

# ─── SIMULACIÓN DE CONSUMO DE API PÚBLICA ───────────────────────────────────
def simular_consumo_api_publica():
    '''
    Simula la verificación de disponibilidad de un endpoint de datos automotrices.
    En entornos de producción, esto conectaría con un servicio web RESTful.
    '''
    url_mock = 'https://jsonplaceholder.typicode.com/posts/1'  # API pública de prueba
    try:
        response = requests.get(url_mock, timeout=5)
        if response.status_code == 200:
            print('[API INFO] Conexión con servicios externos establecida con éxito.')
            print('[API INFO] Estado del servicio:', response.status_code)
        else:
            print('[API WARNING] No se pudo conectar. Usando respaldo local.')
    except Exception as e:
        print(f'[API ERROR] Error de red: {e}. Procediendo con carga local.')

simular_consumo_api_publica()


In [ ]:
# ─── CARGA MASIVA DESDE GOOGLE COLAB ───────────────────────────────────────
file_path = '/content/car_sales_data.csv'

try:
    df = pd.read_csv(file_path)
    print(f'[ÉXITO] Dataset extraído correctamente desde: {file_path}')
    print(f'Total de registros cargados en memoria: {len(df):,}')
    print('\nEstructura inicial de las variables:')
    print(df.dtypes)
    print('\nPrimeras filas:')
    display(df.head())
except FileNotFoundError:
    print(f'[ERROR] Archivo no encontrado en: {file_path}')
    print("Sube el archivo 'car_sales_data.csv' a la sección de archivos de Colab.")


<a id='3'></a>
---
# 3. Hipótesis y Objetivos

## 3.1. Pregunta de Investigación

> ¿Es posible determinar, mediante técnicas de clasificación supervisada, si el **vendedor** constituye un factor relevante para predecir si una venta generará ingresos superiores al percentil 75 del precio de mercado?

## 3.2. Hipótesis Central

> **"Existen especializaciones implícitas entre los vendedores que influyen significativamente en la probabilidad de concretar ventas de alto valor dentro de una concesionaria automotriz."**

## 3.3. Fundamento Conceptual

La hipótesis se sustenta en el concepto de **especialización comercial implícita**: la tendencia —no declarada formalmente— de ciertos asesores de ventas a orientar su actividad hacia segmentos de mayor valor económico, independientemente de las políticas institucionales de asignación de clientes. Esta especialización puede emerger de factores como la experiencia acumulada, la red de contactos, la capacidad de negociación o las preferencias personales por determinadas marcas.

Si esta especialización existe de forma estadísticamente significativa, la variable `Salesperson` debería constituir un predictor relevante dentro de un modelo de clasificación supervisada.

## 3.4. Justificación de la Investigación

- El sector automotriz se caracteriza por transacciones de **alta variabilidad económica** y por la presencia de un factor humano determinante en el proceso de venta.
- Identificar patrones de especialización ocultos en 2.5 M de registros representa un aporte de alto valor para la toma de decisiones gerenciales.
- La aplicación de ML sobre Big Data permite demostrar competencias avanzadas en ingeniería de características y validación estadística de hipótesis empresariales.

## 3.5. Objetivo General

Determinar si la variable `Salesperson` constituye un factor estadísticamente significativo para predecir la probabilidad de concretar una venta de alto valor (≥ percentil 75 del precio de venta), mediante la aplicación y comparación de tres modelos de clasificación supervisada.

## 3.6. Objetivos Específicos

1. Realizar un análisis exploratorio exhaustivo del dataset de 2,500,000 registros, evaluando su estructura, calidad y distribuciones.
2. Detectar y tratar metodológicamente los valores atípicos, nulos e inconsistencias presentes en los datos.
3. Construir la variable binaria objetivo `High_Value_Sale` a partir del umbral del percentil 75.
4. Seleccionar y justificar formalmente las variables predictoras, evitando *data leakage*.
5. Entrenar tres modelos de clasificación: Regresión Logística, Random Forest y XGBoost.
6. Validar la robustez de los modelos mediante validación cruzada.
7. Analizar la importancia de variables del Random Forest para cuantificar el peso de `Salesperson`.
8. Validar o refutar la hipótesis con base en evidencia cuantitativa.


<a id='4'></a>
---
# 4. Análisis Exploratorio de Datos (EDA)

El Análisis Exploratorio de Datos (EDA) constituye una etapa metodológicamente indispensable previa a cualquier proceso de modelado. Su propósito es comprender la estructura, calidad y comportamiento estadístico del dataset **antes** de tomar decisiones de limpieza o de ingeniería de características, evitando así transformaciones arbitrarias no respaldadas por evidencia.

En esta sección se analiza el dataset en su estado **original** (sin modificar). Las acciones de limpieza correspondientes a los hallazgos de esta sección se ejecutan formalmente en la Sección 5.


<a id='41'></a>
## 4.1. Estructura del Dataset

Se examinan las dimensiones, los tipos de dato asignados por pandas y las estadísticas descriptivas básicas, tanto para variables numéricas como categóricas. Este primer acercamiento permite detectar de forma temprana inconsistencias de tipo (por ejemplo, una variable numérica leída como texto) y dimensionar la magnitud computacional del problema.


In [ ]:
if 'df' in locals():
    print('='*70)
    print('  4.1 ESTRUCTURA DEL DATASET')
    print('='*70)

    # ─── DIMENSIONES ─────────────────────────────────────────────────────
    print(f'\nDimensiones del dataset: {df.shape[0]:,} filas  x  {df.shape[1]} columnas')

    # ─── TIPOS DE DATOS ──────────────────────────────────────────────────
    print('\n--- Tipos de datos asignados por pandas ---')
    print(df.dtypes)

    # ─── INFORMACIÓN GENERAL (memoria, nulos por columna) ───────────────
    print('\n--- Información general del DataFrame ---')
    df.info(memory_usage='deep')

    # ─── ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS ─────────────────
    print('\n--- Estadísticas descriptivas: variables numéricas ---')
    display(df.describe(include=[np.number]).T)

    # ─── ESTADÍSTICAS DESCRIPTIVAS — VARIABLES CATEGÓRICAS ───────────────
    print('\n--- Estadísticas descriptivas: variables categóricas ---')
    display(df.describe(include=['object']).T)
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


**Interpretación.** El dataset confirma una escala cercana a los 2.5 millones de registros, lo que justifica las decisiones de optimización computacional adoptadas a lo largo del proyecto (uso de tipos de dato eficientes, muestreo para visualizaciones y validación cruzada sobre submuestras representativas en la Sección 11). La salida de `describe()` sobre las variables categóricas permite además identificar la **cardinalidad** de `Salesperson`, `Car Make` y `Car Model` — un factor relevante para decidir la técnica de codificación en la Sección 7.4.


<a id='42'></a>
## 4.2. Calidad de Datos

Un dataset de origen transaccional —especialmente a escala de millones de registros— suele contener inconsistencias propias de procesos de captura automatizados: duplicados exactos, valores fuera de rango físicamente imposibles, o fechas mal formateadas. Esta subsección audita la calidad del dato **sin modificarlo**, dejando la corrección formal para la Sección 5.


In [ ]:
if 'df' in locals():
    print('='*70)
    print('  4.2 AUDITORÍA DE CALIDAD DE DATOS')
    print('='*70)

    # ─── REGISTROS DUPLICADOS ────────────────────────────────────────────
    n_duplicados = df.duplicated().sum()
    print(f'\n[1] Registros completamente duplicados: {n_duplicados:,} '
          f'({n_duplicados/len(df)*100:.3f}% del total)')

    # ─── VALIDACIÓN DE RANGOS — Car Year ─────────────────────────────────
    anio_actual = pd.Timestamp.now().year
    anio_invalido = df[(df['Car Year'] < 1900) | (df['Car Year'] > anio_actual)]
    print(f'\n[2] Registros con \"Car Year\" fuera de rango plausible (1900-{anio_actual}): '
          f'{len(anio_invalido):,}')

    # ─── VALIDACIÓN DE RANGOS — Sale Price ───────────────────────────────
    precio_invalido = df[pd.to_numeric(df['Sale Price'], errors='coerce') <= 0]
    print(f'[3] Registros con \"Sale Price\" <= 0 (imposible comercialmente): '
          f'{len(precio_invalido):,}')

    # ─── VALIDACIÓN DE RANGOS — Commission Rate ──────────────────────────
    comision_invalida = df[(pd.to_numeric(df['Commission Rate'], errors='coerce') < 0) |
                            (pd.to_numeric(df['Commission Rate'], errors='coerce') > 1)]
    print(f'[4] Registros con \"Commission Rate\" fuera del rango [0, 1]: '
          f'{len(comision_invalida):,}')

    # ─── VERIFICACIÓN DE FECHAS ───────────────────────────────────────────
    fechas_parseadas = pd.to_datetime(df['Date'], errors='coerce')
    fechas_invalidas = fechas_parseadas.isna().sum()
    fechas_futuras = (fechas_parseadas > pd.Timestamp.now()).sum()
    print(f'[5] Fechas no interpretables (formato inválido): {fechas_invalidas:,}')
    print(f'[6] Fechas con valor futuro respecto a hoy: {fechas_futuras:,}')

    # ─── CONSISTENCIA DE TIPOS ────────────────────────────────────────────
    print('\n[7] Tipos de dato actuales vs. tipo esperado:')
    tipos_esperados = {
        'Date': 'fecha', 'Sale Price': 'numérico', 'Commission Rate': 'numérico',
        'Commission Earned': 'numérico', 'Car Year': 'numérico'
    }
    for col, esperado in tipos_esperados.items():
        print(f'    {col:<20} dtype actual: {str(df[col].dtype):<12}  →  esperado: {esperado}')
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


**Interpretación y decisión metodológica.** Los resultados anteriores determinan tres acciones de limpieza que se ejecutarán formalmente en la Sección 5:

1. **Duplicados exactos** → se eliminan en su totalidad (Sección 5.1), dado que no aportan información adicional y sesgarían el entrenamiento al sobre-representar ciertas combinaciones de variables.
2. **Registros con valores fuera de rango** (`Car Year`, `Sale Price`, `Commission Rate`) → se tratan como errores de captura y se excluyen del conjunto de modelado (Sección 5.3), ya que su inclusión introduciría ruido no representativo del fenómeno de negocio bajo estudio.
3. **Fechas no interpretables** → se convierten a `NaT` mediante `errors='coerce'` y se gestionan junto con el resto de valores nulos (Sección 5.2), preservando la integridad de las demás columnas del mismo registro cuando es posible.


<a id='43'></a>
## 4.3. Valores Faltantes

La presencia de valores nulos (`NaN`) puede introducir sesgos severos si no se gestiona adecuadamente: los algoritmos de scikit-learn y XGBoost utilizados en este proyecto no aceptan valores faltantes en las variables predictoras, por lo que su tratamiento es un requisito técnico ineludible, además de una buena práctica metodológica.


In [ ]:
if 'df' in locals():
    print('='*70)
    print('  4.3 ANÁLISIS DE VALORES FALTANTES')
    print('='*70)

    nulos_conteo = df.isnull().sum()
    nulos_pct    = (nulos_conteo / len(df) * 100).round(3)

    tabla_nulos = pd.DataFrame({
        'Conteo de Nulos': nulos_conteo,
        'Porcentaje (%)': nulos_pct
    }).sort_values('Conteo de Nulos', ascending=False)

    print('\nResumen de valores faltantes por columna:')
    display(tabla_nulos)

    # ─── VISUALIZACIÓN ────────────────────────────────────────────────────
    import matplotlib.pyplot as plt
    import seaborn as sns

    cols_con_nulos = tabla_nulos[tabla_nulos['Conteo de Nulos'] > 0]

    fig, ax = plt.subplots(figsize=(10, 5))
    if len(cols_con_nulos) > 0:
        bars = ax.bar(cols_con_nulos.index, cols_con_nulos['Porcentaje (%)'],
                       color='#E74C3C', alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, cols_con_nulos['Porcentaje (%)']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_ylabel('Porcentaje de valores faltantes (%)')
        ax.set_title('Porcentaje de Valores Faltantes por Columna', fontsize=13, fontweight='bold')
        plt.xticks(rotation=30, ha='right')
    else:
        ax.text(0.5, 0.5, 'No se detectaron valores nulos en el dataset',
                ha='center', va='center', fontsize=13)
        ax.set_xticks([]); ax.set_yticks([])
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


**Interpretación y justificación de la estrategia.** Dado que el dataset cuenta con ~2.5 millones de registros, una proporción baja de nulos (habitualmente por debajo del 5%) puede tratarse mediante **eliminación de filas (listwise deletion)** sin comprometer significativamente el poder estadístico del análisis ni introducir sesgo, siempre que la ausencia de datos sea aleatoria (MCAR/MAR) y no esté concentrada en una subpoblación específica de vendedores o marcas.

Se descarta la **imputación** (media, mediana o moda) para las variables utilizadas como predictores categóricos (`Salesperson`, `Car Make`, `Car Model`), ya que imputar un vendedor o una marca inexistente distorsionaría artificialmente la relación que se busca evaluar como parte de la hipótesis central. La estrategia de eliminación de filas con nulos en variables críticas se ejecuta en la Sección 5.2.


<a id='44'></a>
## 4.4. Detección y Tratamiento de Outliers

### ¿Por qué se analizan los outliers?

Los valores atípicos pueden originarse por dos motivos muy distintos, y distinguir entre ambos es la decisión metodológica central de esta subsección:

1. **Errores de captura o medición** (p. ej., un año de fabricación `9999` o una comisión negativa): deben tratarse como ruido y excluirse.
2. **Fenómenos reales y relevantes para el negocio** (p. ej., una venta de USD 150,000 en un mercado donde la mediana es USD 25,000): no son ruido, sino **el propio fenómeno que el proyecto busca modelar**.

### Impacto de los outliers en los modelos de Machine Learning

- **Regresión Logística**: al ser un modelo lineal sensible a la escala, los outliers pueden distorsionar los coeficientes estimados, especialmente si no se aplica un escalador robusto.
- **Random Forest y XGBoost**: al basarse en particiones por umbral, son comparativamente más robustos a outliers, pero una concentración excesiva de valores extremos puede sesgar la importancia relativa de una variable.

### Metodología de detección

Se aplican tres criterios complementarios sobre las principales variables numéricas (`Sale Price`, `Commission Earned`, `Commission Rate`, `Car Year`):

- **Boxplots**: inspección visual de la dispersión y los valores extremos.
- **Método IQR (Rango Intercuartílico)**: un valor se considera atípico si cae fuera de $[Q_1 - 1.5 \\cdot IQR,\\; Q_3 + 1.5 \\cdot IQR]$.
- **Percentiles extremos (P1 / P99)**: como criterio complementario y menos sensible a la forma de la distribución.


In [ ]:
if 'df' in locals():
    import matplotlib.pyplot as plt
    import seaborn as sns

    variables_numericas = ['Sale Price', 'Commission Earned', 'Commission Rate', 'Car Year']

    # Para eficiencia de graficado sobre ~2.5M de registros, se utiliza una
    # muestra aleatoria representativa para las visualizaciones (los cálculos
    # de IQR y percentiles, en cambio, se realizan sobre el dataset COMPLETO).
    df_muestra = df.sample(n=min(100_000, len(df)), random_state=42)

    # ─── BOXPLOTS ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for i, var in enumerate(variables_numericas):
        datos = pd.to_numeric(df_muestra[var], errors='coerce').dropna()
        axes[i].boxplot(datos, vert=True, patch_artist=True,
                         boxprops=dict(facecolor='#3498DB', alpha=0.6))
        axes[i].set_title(var, fontsize=11, fontweight='bold')
        axes[i].grid(axis='y', alpha=0.3)
    fig.suptitle('Boxplots de Variables Numéricas (muestra de 100,000 registros)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ─── MÉTODO IQR (sobre el dataset COMPLETO) ──────────────────────────
    print('='*78)
    print('  4.4 ANÁLISIS DE OUTLIERS — MÉTODO IQR Y PERCENTILES (dataset completo)')
    print('='*78)
    resumen_outliers = []
    for var in variables_numericas:
        datos = pd.to_numeric(df[var], errors='coerce').dropna()
        q1, q3 = datos.quantile(0.25), datos.quantile(0.75)
        iqr = q3 - q1
        lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_outliers_iqr = ((datos < lim_inf) | (datos > lim_sup)).sum()

        p1, p99 = datos.quantile(0.01), datos.quantile(0.99)

        resumen_outliers.append({
            'Variable': var, 'Q1': round(q1, 2), 'Q3': round(q3, 2), 'IQR': round(iqr, 2),
            'Límite Inferior': round(lim_inf, 2), 'Límite Superior': round(lim_sup, 2),
            'Outliers (IQR)': n_outliers_iqr,
            '% Outliers': round(n_outliers_iqr / len(datos) * 100, 2),
            'P1': round(p1, 2), 'P99': round(p99, 2)
        })

    tabla_outliers = pd.DataFrame(resumen_outliers)
    display(tabla_outliers)
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


**Decisión metodológica adoptada (justificación detallada):**

| Variable | Tratamiento | Justificación |
|---|---|---|
| `Sale Price` | **Se conservan los outliers** | Las ventas en el extremo superior de la distribución son, por definición, el fenómeno de interés del proyecto: la variable objetivo `High_Value_Sale` (Sección 6) se construye precisamente a partir del percentil 75 de esta variable. Eliminarlas equivaldría a remover la señal que se busca predecir, invalidando la hipótesis. |
| `Commission Earned` | **Se conservan** | Es función directa de `Sale Price`; sus valores extremos son consistentes con ventas de alto valor, no con errores de captura. (Además, esta variable se excluye del modelado por *data leakage* — ver Sección 7.2). |
| `Commission Rate` | **Se eliminan los valores fuera de [0, 1]** | Un porcentaje de comisión negativo o superior al 100% no tiene interpretación de negocio válida; constituye un error de captura, no una observación legítima. |
| `Car Year` | **Se eliminan los valores fuera de [1900, año actual]** | Un año de fabricación anterior a 1900 o posterior al año en curso es físicamente imposible y corresponde a errores de digitación. |

En síntesis, la estrategia adoptada **no aplica un criterio único de poda automática (p. ej. recorte ciego por IQR) sobre todas las variables**, sino que evalúa cada caso según su origen probable (error de captura vs. fenómeno real), priorizando la preservación de la señal estadística relevante para la hipótesis de negocio.


<a id='45'></a>
## 4.5. Distribuciones de Variables

Se examina la forma de la distribución de las principales variables numéricas y categóricas, lo cual orienta tanto la elección de transformaciones (p. ej. escalado en la Regresión Logística) como la interpretación posterior de la importancia de variables.


In [ ]:
if 'df' in locals():
    import matplotlib.pyplot as plt
    import seaborn as sns

    df_muestra = df.sample(n=min(200_000, len(df)), random_state=42)

    # ─── DISTRIBUCIONES NUMÉRICAS ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.histplot(pd.to_numeric(df_muestra['Car Year'], errors='coerce').dropna(),
                 bins=30, ax=axes[0], color='#3498DB', kde=False)
    axes[0].set_title('Distribución: Car Year', fontweight='bold')

    sns.histplot(pd.to_numeric(df_muestra['Sale Price'], errors='coerce').dropna(),
                 bins=50, ax=axes[1], color='#2ECC71', kde=True)
    axes[1].set_title('Distribución: Sale Price', fontweight='bold')

    sns.histplot(pd.to_numeric(df_muestra['Commission Earned'], errors='coerce').dropna(),
                 bins=50, ax=axes[2], color='#E67E22', kde=True)
    axes[2].set_title('Distribución: Commission Earned', fontweight='bold')

    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ─── DISTRIBUCIONES CATEGÓRICAS — TOP N ───────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    top_vendedores = df['Salesperson'].value_counts().head(10)
    axes[0].barh(top_vendedores.index[::-1], top_vendedores.values[::-1], color='#9B59B6')
    axes[0].set_title('Top 10 Vendedores por N° de Ventas', fontweight='bold')

    top_marcas = df['Car Make'].value_counts().head(10)
    axes[1].barh(top_marcas.index[::-1], top_marcas.values[::-1], color='#1ABC9C')
    axes[1].set_title('Top 10 Marcas por N° de Ventas', fontweight='bold')

    top_modelos = df['Car Model'].value_counts().head(10)
    axes[2].barh(top_modelos.index[::-1], top_modelos.values[::-1], color='#F39C12')
    axes[2].set_title('Top 10 Modelos por N° de Ventas', fontweight='bold')

    for ax in axes:
        ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


**Interpretación.**

- **`Car Year`** muestra una distribución concentrada en los años recientes, consistente con un parque automotor de rotación comercial activa.
- **`Sale Price`** presenta una asimetría positiva (cola hacia la derecha) típica de variables de precio: la mayoría de las ventas se concentran en un rango medio, con una minoría de transacciones de alto valor que constituyen precisamente el segmento de interés (`High_Value_Sale`).
- **`Commission Earned`** replica la forma de `Sale Price`, confirmando su relación funcional directa — evidencia adicional que respalda su exclusión del modelo por *data leakage* (Sección 7.2).
- Los gráficos de **Top 10** evidencian que tanto vendedores como marcas y modelos no están uniformemente distribuidos en el volumen de ventas, lo cual es consistente con la posibilidad de que existan patrones de especialización — el fenómeno central que la hipótesis busca confirmar.


<a id='5'></a>
---
# 5. Limpieza y Preprocesamiento

A partir de los hallazgos documentados en la Sección 4, se ejecutan a continuación las acciones de limpieza correspondientes sobre el DataFrame `df`. Cada subsección corresponde a una decisión metodológica ya justificada previamente.


<a id='51'></a>
## 5.1. Eliminación de Duplicados

Se eliminan los registros completamente duplicados detectados en la Sección 4.2, ya que representan observaciones redundantes que sobre-representarían artificialmente ciertas combinaciones de variables durante el entrenamiento.


In [ ]:
if 'df' in locals():
    filas_antes = df.shape[0]
    df = df.drop_duplicates()
    filas_despues = df.shape[0]
    print(f'Registros antes de la limpieza : {filas_antes:,}')
    print(f'Registros duplicados eliminados: {filas_antes - filas_despues:,}')
    print(f'Registros después de la limpieza: {filas_despues:,}')
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


<a id='52'></a>
## 5.2. Tratamiento de Valores Nulos

Conforme a la estrategia justificada en la Sección 4.3, se eliminan (listwise deletion) los registros con valores nulos en las columnas críticas para el análisis y el modelado. Dado el tamaño del dataset (millones de registros) y la baja proporción esperada de nulos, esta estrategia no compromete la representatividad estadística de la muestra resultante.


In [ ]:
if 'df' in locals():
    columnas_criticas = ['Date', 'Salesperson', 'Car Make', 'Car Model',
                          'Car Year', 'Sale Price']

    filas_antes = df.shape[0]
    print('Conteo de nulos en columnas críticas (antes del tratamiento):')
    print(df[columnas_criticas].isnull().sum())

    df = df.dropna(subset=columnas_criticas)
    filas_despues = df.shape[0]

    print(f'\nRegistros eliminados por nulos en columnas críticas: {filas_antes - filas_despues:,}')
    print(f'Registros restantes: {filas_despues:,} '
          f'({filas_despues/filas_antes*100:.2f}% del total previo)')
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


<a id='53'></a>
## 5.3. Conversión de Tipos y Validación de Rangos

Se estandarizan los tipos de dato (`Date` a `datetime64`, variables numéricas a `float`/`int`) y se eliminan los registros con valores fuera de rango identificados en la Sección 4.2 (`Car Year`, `Sale Price`, `Commission Rate`), tratándolos como errores de captura.


In [ ]:
if 'df' in locals():
    # ─── CONVERSIÓN DE TIPOS ───────────────────────────────────────────────
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Sale Price']        = pd.to_numeric(df['Sale Price'], errors='coerce')
    df['Commission Earned'] = pd.to_numeric(df['Commission Earned'], errors='coerce')
    df['Commission Rate']   = pd.to_numeric(df['Commission Rate'], errors='coerce')
    df['Car Year']          = pd.to_numeric(df['Car Year'], errors='coerce')

    filas_antes = df.shape[0]

    # ─── VALIDACIÓN DE RANGOS (eliminación de errores de captura) ─────────
    anio_actual = pd.Timestamp.now().year
    df = df[df['Date'].notna()]
    df = df[(df['Car Year'] >= 1900) & (df['Car Year'] <= anio_actual)]
    df = df[df['Sale Price'] > 0]
    df = df[(df['Commission Rate'] >= 0) & (df['Commission Rate'] <= 1)]

    filas_despues = df.shape[0]
    print(f'Registros eliminados por valores fuera de rango: {filas_antes - filas_despues:,}')
    print(f'Registros restantes tras validación de rangos  : {filas_despues:,}')

    print('\n[COMPLETADO] Tipos de datos estandarizados:')
    print(df[['Date', 'Sale Price', 'Commission Rate', 'Commission Earned', 'Car Year']].dtypes)
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


<a id='54'></a>
## 5.4. Ingeniería de Características Temporales

A partir de `Date` se extraen variables derivadas que capturan patrones de **estacionalidad comercial**: `Month` (mes calendario), `Quarter` (trimestre), `ISO_Week` (semana ISO) y `Day_of_Week` (día de la semana). Estas variables permiten que los modelos detecten ciclos de demanda recurrentes (por ejemplo, picos de venta de fin de año o de cierre de trimestre fiscal) sin necesidad de codificar la fecha completa, que tendría una cardinalidad excesivamente alta para ser útil como predictor directo.


In [ ]:
if 'df' in locals():
    df['Month']       = df['Date'].dt.month
    df['Quarter']     = df['Date'].dt.quarter
    df['ISO_Week']    = df['Date'].dt.isocalendar().week
    df['Day_of_Week'] = df['Date'].dt.dayofweek

    print('[COMPLETADO] Variables temporales derivadas correctamente.')
    print('\nEstructura final del dataset preprocesado:')
    display(df[['Date', 'Car Make', 'Month', 'Quarter', 'ISO_Week',
                'Day_of_Week', 'Sale Price']].head())
    print(f'\nDimensiones finales del dataset limpio: {df.shape}')
else:
    print('El dataframe no está cargado. Ejecuta primero la Sección 2.')


<a id='6'></a>
---
# 6. Construcción de la Variable Objetivo

## 6.1. Definición

Para transformar el problema de negocio en una tarea de **clasificación supervisada**, se construye la variable binaria `High_Value_Sale`:

$$
\text{High\_Value\_Sale} = \begin{cases} 1 & \text{si } \texttt{Sale Price} \geq P_{75} \\ 0 & \text{si } \texttt{Sale Price} < P_{75} \end{cases}
$$

## 6.2. Justificación Académica del Umbral

La selección del **percentil 75** como umbral responde a tres criterios metodológicos:

1. **Robustez estadística**: el P75 es resistente a valores extremos, garantizando que el umbral refleje la distribución real del mercado (a diferencia de la media, que es sensible a la asimetría observada en la Sección 4.5).
2. **Balance de clases**: se obtiene una distribución natural del 75% en clase 0 (venta regular) y 25% en clase 1 (venta de alto valor), gestionable mediante `class_weight='balanced'` en los modelos.
3. **Relevancia comercial**: las ventas en el cuartil superior representan el mayor impacto económico para la concesionaria, siendo el segmento de mayor interés para la toma de decisiones gerenciales.


In [ ]:
if 'df' in locals():
    umbral_p75 = df['Sale Price'].quantile(0.75)
    print(f'Umbral del Percentil 75 (P75): USD {umbral_p75:,.2f}')

    df['High_Value_Sale'] = (df['Sale Price'] >= umbral_p75).astype(int)

    print('\nDistribución de clases (absoluta):')
    print(df['High_Value_Sale'].value_counts())
    print('\nDistribución de clases (porcentual):')
    print(df['High_Value_Sale'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
else:
    print('El dataframe no está cargado. Ejecuta primero las secciones anteriores.')


## 6.3. Visualización de la Variable Objetivo


In [ ]:
if 'df' in locals() and 'High_Value_Sale' in df.columns:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ─── HISTOGRAMA DE SALE PRICE CON LÍNEA DEL P75 ────────────────────────
    df_muestra = df.sample(n=min(200_000, len(df)), random_state=42)
    sns.histplot(df_muestra['Sale Price'], bins=60, ax=axes[0], color='#3498DB', kde=True)
    axes[0].axvline(umbral_p75, color='#E74C3C', linestyle='--', linewidth=2,
                     label=f'P75 = USD {umbral_p75:,.0f}')
    axes[0].set_title('Distribución de Sale Price con Umbral P75', fontweight='bold')
    axes[0].set_xlabel('Sale Price (USD)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # ─── DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ──────────────────────────────
    conteo_target = df['High_Value_Sale'].value_counts().sort_index()
    etiquetas = ['Venta Regular (0)', 'Alto Valor (1)']
    colores_target = ['#95A5A6', '#E74C3C']
    bars = axes[1].bar(etiquetas, conteo_target.values, color=colores_target, alpha=0.85)
    for bar, val in zip(bars, conteo_target.values):
        pct = val / conteo_target.sum() * 100
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                      f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
    axes[1].set_title('Distribución de la Variable Objetivo: High_Value_Sale', fontweight='bold')
    axes[1].set_ylabel('N° de registros')
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('Ejecuta primero la celda de creación de la variable objetivo.')


**Interpretación.** El histograma confirma visualmente la asimetría positiva de `Sale Price` identificada en la Sección 4.5, y ubica el umbral P75 en la zona donde la densidad de la distribución comienza a decrecer marcadamente — consistente con la idea de que las ventas de alto valor son, en efecto, una minoría estadística (~25%) pero económicamente significativa. El gráfico de barras confirma el balance de clases esperado (75/25), lo cual valida la decisión de utilizar `class_weight='balanced'` en los modelos de la Sección 9 en lugar de aplicar técnicas de remuestreo más invasivas (SMOTE, undersampling), que no son necesarias ante un desbalance moderado.

## 6.4. Prevención de Data Leakage

Se excluyen deliberadamente del conjunto de predictores las variables `Commission Earned` y `Commission Rate`, ya que ambas están matemática o funcionalmente determinadas por `Sale Price` (la variable a partir de la cual se construye el target). Su inclusión produciría métricas artificialmente infladas que no reflejarían la capacidad real de generalización del modelo. Esta decisión se justifica formalmente en la Sección 7.2.


<a id='7'></a>
---
# 7. Selección y Justificación de Variables

La selección de variables predictoras no es una decisión arbitraria: cada variable incluida en el modelo debe aportar valor predictivo genuino y estar disponible **antes** del momento en que se necesitaría predecir el resultado (es decir, no debe filtrar información del futuro o del propio resultado). Esta sección documenta formalmente esa justificación, variable por variable.


<a id='71'></a>
## 7.1. Variables Predictoras Seleccionadas

| Variable | Tipo | Justificación |
|---|---|---|
| `Salesperson` | Categórica | Variable central de la hipótesis |
| `Car Make` | Categórica | Fabricante: correlaciona con segmento de precio |
| `Car Model` | Categórica | Modelo: determina la gama del vehículo |
| `Car Year` | Numérica Discreta | Año: relacionado con valor y depreciación |
| `Month` | Numérica Discreta | Mes: captura efectos de temporalidad |
| `Quarter` | Numérica Discreta | Trimestre: agrupa estacionalidad |

### `Salesperson`

Es la variable **más importante del proyecto**, ya que la hipótesis central postula directamente que el vendedor influye en la probabilidad de concretar una venta de alto valor. Aporta al modelo la posibilidad de capturar diferencias individuales en habilidad comercial, especialización por segmento o estilo de negociación que no están explicadas por las características del propio vehículo. Si esta variable resulta relevante en el modelo (Sección 12), constituye evidencia directa a favor de la hipótesis; si no lo es, constituye evidencia en contra.

### `Car Make`

El fabricante del vehículo está intrínsecamente relacionado con el **segmento económico** del producto: marcas premium tienden a posicionarse en rangos de precio superiores a marcas de volumen masivo. Esta variable permite al modelo distinguir entre ventas de alto valor explicadas por el **producto** (qué se vendió) frente a ventas de alto valor explicadas por el **vendedor** (quién lo vendió) — la distinción que sustenta toda la pregunta de investigación.

### `Car Model`

Aporta un **nivel de detalle más granular** que `Car Make`: dentro de una misma marca, distintos modelos (compacto vs. SUV, gama básica vs. gama alta) tienen relación directa con el precio de venta. Incluir esta variable permite al modelo capturar variabilidad de precio que `Car Make` por sí solo no explicaría completamente.

### `Car Year`

El año de fabricación está directamente relacionado con la **depreciación** del vehículo: en términos generales, los vehículos más nuevos retienen mayor valor de mercado. Esta variable permite controlar el efecto de la antigüedad del vehículo sobre el precio, de modo que la posible influencia de `Salesperson` no se confunda con el simple hecho de haber vendido vehículos más nuevos.

### `Month` y `Quarter`

Capturan **estacionalidad** y **comportamiento comercial trimestral**: los patrones de demanda automotriz suelen presentar variaciones cíclicas (por ejemplo, cierres de año fiscal, promociones de temporada o lanzamientos de nuevos modelos). Incluir estas variables evita que el modelo le atribuya a `Salesperson` un efecto que en realidad corresponde a un patrón temporal compartido por todos los vendedores.


<a id='72'></a>
## 7.2. Variables Excluidas del Modelo

| Variable | Razón de exclusión |
|---|---|
| `Commission Earned` | Data Leakage (función directa de `Sale Price`) |
| `Commission Rate` | Data Leakage indirecto |
| `Customer Name` | Sin valor predictivo / alta cardinalidad |
| `Sale Price` | Es la variable origen del target (leakage directo y total) |

### ¿Qué es Data Leakage?

El **data leakage** (fuga de datos) ocurre cuando una variable predictora contiene información que, en la práctica, no estaría disponible en el momento de realizar la predicción, o que está matemáticamente derivada de la variable objetivo. Un modelo entrenado con variables filtradas obtiene métricas de evaluación artificialmente altas durante las pruebas, pero falla en producir predicciones útiles en un escenario real, porque "memoriza" la respuesta en lugar de aprender el patrón subyacente.

### `Commission Earned` — Leakage directo

Esta variable se calcula típicamente como `Sale Price × Commission Rate`. Es, en esencia, una transformación matemática de la variable objetivo. Incluirla sería equivalente a darle al modelo la respuesta junto con la pregunta.

### `Commission Rate` — Leakage indirecto

En muchas estructuras comerciales, el porcentaje de comisión asignado varía según el monto de la venta (por ejemplo, tasas preferenciales para ventas de mayor valor). Aunque la relación es menos directa que con `Commission Earned`, su inclusión introduce el mismo riesgo de inflar artificialmente el desempeño del modelo.

### `Customer Name` — Falta de valor predictivo y alta cardinalidad

Al tratarse de un identificador casi único por transacción (cardinalidad extremadamente alta, cercana al número total de registros), esta variable no captura ningún patrón generalizable: codificarla numéricamente induciría al modelo a **sobreajustarse (overfitting)**, memorizando clientes específicos en lugar de aprender relaciones estructurales entre las variables del negocio.

### `Sale Price` — Leakage total (variable origen)

`Sale Price` es la variable a partir de la cual se construye directamente `High_Value_Sale` (Sección 6.1). Incluirla como predictor sería metodológicamente inválido: el modelo simplemente aprendería a comparar el valor contra el umbral, sin realizar ninguna inferencia real. Esta variable es, por definición, inutilizable como feature en este problema.

### Sobreajuste (Overfitting) — concepto transversal

El **sobreajuste** ocurre cuando un modelo aprende patrones específicos del conjunto de entrenamiento (incluyendo ruido o identificadores únicos) en lugar de relaciones generalizables. Las variables excluidas en esta sección —especialmente `Customer Name`— incrementarían drásticamente el riesgo de overfitting sin aportar capacidad predictiva genuina sobre nuevos datos.


<a id='73'></a>
## 7.3. Evidencia Visual de la Relación con la Variable Objetivo

Antes de proceder al modelado, se examina visualmente si las variables seleccionadas muestran asociación empírica con `High_Value_Sale`, lo cual respalda (o cuestiona) la justificación teórica presentada en 7.1.


In [ ]:
if 'df' in locals() and 'High_Value_Sale' in df.columns:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    # ─── (1) TASA DE ALTO VALOR POR VENDEDOR (TOP 15 POR VOLUMEN) ──────────
    top15_vendedores = df['Salesperson'].value_counts().head(15).index
    tasa_vendedor = (df[df['Salesperson'].isin(top15_vendedores)]
                      .groupby('Salesperson')['High_Value_Sale'].mean()
                      .sort_values(ascending=False) * 100)
    axes[0, 0].barh(tasa_vendedor.index[::-1], tasa_vendedor.values[::-1], color='#E74C3C', alpha=0.85)
    axes[0, 0].axvline(df['High_Value_Sale'].mean() * 100, color='black', linestyle='--',
                        label='Promedio general')
    axes[0, 0].set_title('% de Ventas de Alto Valor por Vendedor (Top 15 por volumen)', fontweight='bold')
    axes[0, 0].set_xlabel('% Ventas de Alto Valor')
    axes[0, 0].legend()

    # ─── (2) TASA DE ALTO VALOR POR MARCA ──────────────────────────────────
    tasa_marca = (df.groupby('Car Make')['High_Value_Sale'].mean()
                  .sort_values(ascending=False) * 100)
    axes[0, 1].barh(tasa_marca.index[::-1], tasa_marca.values[::-1], color='#3498DB', alpha=0.85)
    axes[0, 1].axvline(df['High_Value_Sale'].mean() * 100, color='black', linestyle='--',
                        label='Promedio general')
    axes[0, 1].set_title('% de Ventas de Alto Valor por Car Make', fontweight='bold')
    axes[0, 1].set_xlabel('% Ventas de Alto Valor')
    axes[0, 1].legend()

    # ─── (3) CAR YEAR POR CLASE OBJETIVO (BOXPLOT) ─────────────────────────
    df_muestra = df.sample(n=min(150_000, len(df)), random_state=42)
    sns.boxplot(data=df_muestra, x='High_Value_Sale', y='Car Year', ax=axes[1, 0],
                palette=['#95A5A6', '#E74C3C'])
    axes[1, 0].set_xticklabels(['Venta Regular (0)', 'Alto Valor (1)'])
    axes[1, 0].set_title('Car Year según Clase Objetivo', fontweight='bold')

    # ─── (4) VENTAS DE ALTO VALOR POR TRIMESTRE (COUNTPLOT AGRUPADO) ───────
    tabla_q = pd.crosstab(df['Quarter'], df['High_Value_Sale'], normalize='index') * 100
    tabla_q.plot(kind='bar', ax=axes[1, 1], color=['#95A5A6', '#E74C3C'], alpha=0.85)
    axes[1, 1].set_title('% de Ventas de Alto Valor por Trimestre', fontweight='bold')
    axes[1, 1].set_xlabel('Quarter')
    axes[1, 1].set_ylabel('% dentro del trimestre')
    axes[1, 1].legend(['Regular (0)', 'Alto Valor (1)'])
    axes[1, 1].tick_params(axis='x', rotation=0)

    for ax in axes.flat:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ─── HEATMAP DE CORRELACIÓN (VARIABLES NUMÉRICAS + TARGET) ─────────────
    fig, ax = plt.subplots(figsize=(7, 5.5))
    cols_corr = ['Car Year', 'Month', 'Quarter', 'High_Value_Sale']
    matriz_corr = df[cols_corr].corr()
    sns.heatmap(matriz_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                ax=ax, square=True, linewidths=0.5)
    ax.set_title('Matriz de Correlación — Variables Numéricas y Target', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Ejecuta primero la Sección 6 para generar la variable objetivo.')


**Interpretación.** Los gráficos de tasa de alto valor por vendedor y por marca muestran **dispersión visible alrededor del promedio general** (línea punteada): algunos vendedores y algunas marcas concentran una proporción de ventas de alto valor notablemente superior o inferior al resto, lo cual constituye evidencia exploratoria preliminar a favor de la hipótesis de especialización. El boxplot de `Car Year` por clase objetivo y el comportamiento por `Quarter` confirman que estas variables aportan información discriminante adicional, complementaria a `Salesperson`. La matriz de correlación, al tratarse de variables predominantemente categóricas o de baja varianza lineal, muestra coeficientes moderados — esperable, dado que la relación principal del proyecto (`Salesperson` vs. target) es categórica y se evalúa con mayor propiedad mediante el análisis de importancia de variables del Random Forest (Sección 12), no mediante correlación lineal de Pearson.


<a id='74'></a>
## 7.4. Codificación de Variables Categóricas

Las variables categóricas (`Salesperson`, `Car Make`, `Car Model`) se transforman a representación numérica mediante **Label Encoding**, técnica adecuada para los modelos basados en árboles (Random Forest, XGBoost) utilizados en este proyecto, ya que estos no asumen una relación de orden entre las categorías codificadas. Para la Regresión Logística, este efecto se mitiga mediante el escalado posterior de las variables (`StandardScaler`, Sección 9.2).


In [ ]:
from sklearn.preprocessing import LabelEncoder

if 'df' in locals() and 'High_Value_Sale' in df.columns:
    # ─── DEFINICIÓN DE FEATURES Y TARGET ────────────────────────────────────
    features = ['Salesperson', 'Car Make', 'Car Model', 'Car Year', 'Month', 'Quarter']
    target   = 'High_Value_Sale'

    df_model = df[features + [target]].dropna().copy()
    print(f'Registros disponibles para modelado: {len(df_model):,}')

    # ─── CODIFICACIÓN DE VARIABLES CATEGÓRICAS ───────────────────────────────
    le = LabelEncoder()
    for col in ['Salesperson', 'Car Make', 'Car Model']:
        df_model[col] = le.fit_transform(df_model[col].astype(str))
    print('[OK] Codificación Label Encoding aplicada a variables categóricas.')

    X = df_model[features]
    y = df_model[target]
    print(f'\nDimensiones de X: {X.shape}')
    print(f'Distribución del target:\n{y.value_counts().to_string()}')
else:
    print('Asegúrate de haber ejecutado las secciones anteriores correctamente.')


<a id='8'></a>
---
# 8. División de Entrenamiento y Prueba

## 8.1. Conceptos Fundamentales

Antes de entrenar cualquier modelo, el dataset debe dividirse en dos subconjuntos disjuntos:

- **Conjunto de Entrenamiento (Training Set)**: es la porción de los datos que el modelo utiliza para **aprender** los patrones y relaciones entre las variables predictoras y la variable objetivo. Durante esta etapa, el algoritmo ajusta sus parámetros internos (coeficientes en la Regresión Logística, particiones en los árboles de Random Forest y XGBoost) para minimizar el error sobre estos datos.

- **Conjunto de Prueba (Test Set)**: es la porción de los datos que el modelo **nunca observa durante el entrenamiento**. Se utiliza exclusivamente al final, para evaluar qué tan bien generaliza el modelo a datos nuevos y no vistos.

## 8.2. ¿Por qué los datos de prueba nunca deben utilizarse durante el entrenamiento?

Si el modelo tuviera acceso, aunque sea parcial, a los datos de prueba durante el entrenamiento, las métricas de evaluación (accuracy, F1, ROC-AUC) estarían midiendo la capacidad del modelo para **recordar** información ya vista, no su capacidad real de generalizar. Esto produciría una falsa sensación de buen desempeño que se desmoronaría en un escenario de producción real, donde el modelo enfrenta exclusivamente datos nuevos. Mantener una separación estricta entre ambos conjuntos es, por tanto, la única forma de obtener una estimación honesta del desempeño del modelo.

## 8.3. ¿Qué es el Overfitting (Sobreajuste)?

El sobreajuste ocurre cuando un modelo aprende excesivamente los detalles y el ruido específicos del conjunto de entrenamiento, en lugar de capturar el patrón general subyacente. Un modelo sobreajustado obtiene métricas excelentes sobre los datos de entrenamiento, pero un desempeño marcadamente inferior sobre datos nuevos —exactamente lo que el conjunto de prueba permite detectar. La comparación sistemática entre el desempeño en entrenamiento y en prueba (y, de forma más robusta, la validación cruzada de la Sección 11) es la herramienta principal para diagnosticar overfitting.

## 8.4. Capacidad de Generalización

La **generalización** es la capacidad de un modelo de mantener un desempeño consistente ante datos que no formaron parte de su proceso de aprendizaje. Es, en última instancia, el objetivo real de cualquier proyecto de Machine Learning aplicado: un modelo que predice perfectamente el pasado pero falla en el futuro no tiene utilidad práctica para la toma de decisiones gerenciales que motiva este proyecto.

## 8.5. Metodología de Partición Adoptada: 70% / 30%

Siguiendo el lineamiento metodológico del curso, se reserva un **30% de los datos para el conjunto de prueba**, en lugar de la división 80/20 utilizada en versiones preliminares del proyecto:

```python
train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
```

### Justificación de la proporción 70/30 para un dataset de 2.5 millones de registros

1. **Tamaño absoluto suficiente en ambos conjuntos**: con ~2.5 millones de registros, una reserva del 30% para prueba todavía representa más de 700,000 observaciones — una muestra más que suficiente para obtener estimaciones de desempeño estadísticamente estables y de baja varianza, mientras que el 70% restante (~1.75 millones de registros) sigue siendo ampliamente suficiente para que algoritmos como Random Forest y XGBoost aprendan patrones robustos.
2. **Mayor confianza estadística en la evaluación**: a mayor tamaño del conjunto de prueba, menor es el margen de error de las métricas reportadas (accuracy, F1, ROC-AUC), lo cual es especialmente relevante en un proyecto cuya conclusión final depende de comparar el desempeño de tres modelos distintos.
3. **Bajo riesgo de subentrenamiento**: a diferencia de datasets pequeños (cientos o miles de registros), donde reservar un 30-40% para prueba podría privar al modelo de datos de entrenamiento necesarios, en un dataset de millones de registros esta cantidad de "datos sacrificados" no compromete la capacidad de aprendizaje del modelo.

### Estratificación (`stratify=y`)

Se utiliza partición **estratificada**, lo que garantiza que la proporción de clases (75% ventas regulares / 25% ventas de alto valor, según la Sección 6) se mantenga **idéntica** tanto en el conjunto de entrenamiento como en el de prueba. Sin estratificación, una partición puramente aleatoria podría, por azar, generar un conjunto de prueba con una proporción de clases distinta a la real, distorsionando la evaluación de métricas sensibles al desbalance como Precision, Recall y F1.


In [ ]:
from sklearn.model_selection import train_test_split

if 'X' in locals() and 'y' in locals():
    # ─── PARTICIÓN ESTRATIFICADA 70/30 ───────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )

    print('='*60)
    print('  PARTICIÓN DEL DATASET — 70% Entrenamiento / 30% Prueba')
    print('='*60)
    print(f'\nConjunto de Entrenamiento: {X_train.shape[0]:,} registros ({X_train.shape[0]/len(X)*100:.1f}%)')
    print(f'Conjunto de Prueba       : {X_test.shape[0]:,} registros ({X_test.shape[0]/len(X)*100:.1f}%)')

    print('\nDistribución del target — Entrenamiento:')
    print(y_train.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
    print('\nDistribución del target — Prueba:')
    print(y_test.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
else:
    print('Asegúrate de haber ejecutado la Sección 7.4 correctamente.')


<a id='9'></a>
---
# 9. Entrenamiento de Modelos

Se entrenan tres modelos de clasificación supervisada de **complejidad creciente**, lo cual permite evaluar la relación entre la sofisticación del algoritmo y la ganancia real en capacidad predictiva: **Regresión Logística** (línea base interpretable), **Random Forest** (ensamble robusto con importancia de variables nativa) y **XGBoost** (máxima capacidad predictiva mediante boosting).


<a id='91'></a>
## 9.1. Función de Evaluación Unificada

Para garantizar que los tres modelos se evalúen bajo criterios idénticos y comparables, se define una función única que calcula el conjunto completo de métricas relevantes para un problema de clasificación binaria desbalanceada:

| Métrica | Descripción |
|---|---|
| **Accuracy** | Proporción de predicciones correctas sobre el total |
| **Precision** | De las ventas predichas como alto valor, ¿cuántas realmente lo son? |
| **Recall** | De todas las ventas reales de alto valor, ¿cuántas detecta el modelo? |
| **F1 Score** | Media armónica de Precision y Recall |
| **ROC-AUC** | Capacidad discriminante del modelo en todos los umbrales posibles |
| **Matriz de Confusión** | Desglose de Verdaderos/Falsos Positivos y Negativos |


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve)

resultados_globales = {}  # Diccionario para almacenar métricas de todos los modelos

def evaluar_modelo(nombre, modelo, X_test, y_test):
    '''
    Evalúa un modelo de clasificación y retorna un diccionario con todas las métricas.
    También grafica la Matriz de Confusión y la Curva ROC.
    '''
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)

    sep = '=' * 55
    print(f'\n{sep}')
    print(f'  RESULTADOS: {nombre}')
    print(f'{sep}')
    print(f'  Accuracy  : {acc*100:.2f}%')
    print(f'  Precision : {prec*100:.2f}%')
    print(f'  Recall    : {rec*100:.2f}%')
    print(f'  F1 Score  : {f1*100:.2f}%')
    print(f'  ROC-AUC   : {auc*100:.2f}%')
    print(f'{sep}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Venta Regular (0)', 'Alto Valor (1)'])
    disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title(f'Matriz de Confusión\n{nombre}', fontsize=13, fontweight='bold')

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC-AUC = {auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador Aleatorio')
    axes[1].set_xlabel('Tasa de Falsos Positivos')
    axes[1].set_ylabel('Tasa de Verdaderos Positivos')
    axes[1].set_title(f'Curva ROC\n{nombre}', fontsize=13, fontweight='bold')
    axes[1].legend(loc='lower right')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    metricas = {
        'Accuracy' : acc,
        'Precision': prec,
        'Recall'   : rec,
        'F1 Score' : f1,
        'ROC-AUC'  : auc
    }
    resultados_globales[nombre] = metricas
    return metricas

print('[OK] Función de evaluación definida y lista para usar.')


<a id='92'></a>
## 9.2. Modelo 1: Regresión Logística

### Descripción Técnica
La Regresión Logística es un algoritmo de clasificación lineal que estima la probabilidad de pertenencia a una clase mediante la **función sigmoide** aplicada a una combinación lineal ponderada de las variables predictoras:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \cdots + \beta_n X_n)}}$$

### Justificación de Selección
La Regresión Logística cumple el rol de **modelo de línea base (baseline)** en el experimento. Su principal aporte es la **interpretabilidad de sus coeficientes**: si el coeficiente asociado a `Salesperson` resulta de magnitud relevante en comparación con `Car Make` o `Car Model`, esto constituye evidencia inicial a favor de la hipótesis. Su bajo costo computacional la hace viable para el procesamiento de millones de registros.

**Variables utilizadas:** `Salesperson`, `Car Make`, `Car Model`, `Car Year`, `Month`, `Quarter`
**Variable objetivo:** `High_Value_Sale`
**Nota:** Se aplica `StandardScaler` dentro de un `Pipeline` para normalizar las features, requisito relevante ya que la Regresión Logística es sensible a la escala de las variables.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ─── MODELO 1: REGRESIÓN LOGÍSTICA ──────────────────────────────────────────
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # Compensa el desbalance de clases
        solver='lbfgs'
    ))
])

print('Entrenando Regresión Logística...')
pipeline_lr.fit(X_train, y_train)
print('[OK] Entrenamiento completado.')

# Evaluación
resultados_lr = evaluar_modelo('Regresión Logística', pipeline_lr, X_test, y_test)


<a id='93'></a>
## 9.3. Modelo 2: Random Forest Classifier

### Descripción Técnica
El Random Forest es un método de ensamble basado en la construcción de múltiples árboles de decisión entrenados sobre submuestras aleatorias del dataset (**bagging**) con selección aleatoria de features en cada división (**feature randomness**). La clasificación final se obtiene por **votación mayoritaria** de todos los árboles.

Esta doble aleatorización reduce la varianza del modelo y lo hace robusto frente al overfitting, lo cual es especialmente valioso con variables categóricas de alta cardinalidad como `Salesperson`.

### Justificación de Selección
El Random Forest ocupa el **rol central del experimento** debido a su capacidad nativa para calcular la **importancia de variables** mediante la reducción media de impureza de Gini. Esta característica permite evaluar directamente si `Salesperson` figura como uno de los predictores más determinantes, siendo la evidencia cuantitativa más directa para validar la hipótesis.

**Variables utilizadas:** `Salesperson`, `Car Make`, `Car Model`, `Car Year`, `Month`, `Quarter`
**Variable objetivo:** `High_Value_Sale`


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ─── MODELO 2: RANDOM FOREST ────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1  # Usa todos los núcleos disponibles
)

print('Entrenando Random Forest (200 árboles)...')
rf.fit(X_train, y_train)
print('[OK] Entrenamiento completado.')

# Evaluación
resultados_rf = evaluar_modelo('Random Forest', rf, X_test, y_test)


<a id='94'></a>
## 9.4. Modelo 3: XGBoost / Gradient Boosting Classifier

### Descripción Técnica
XGBoost (Extreme Gradient Boosting) es un algoritmo de ensamble basado en **boosting secuencial**: cada árbol sucesivo se entrena para corregir los errores residuales del árbol anterior, minimizando una función de pérdida logística mediante **descenso de gradiente**. A diferencia del Random Forest, el boosting reduce el **sesgo** en lugar de la varianza, lo que lo convierte en el algoritmo de mayor capacidad predictiva de los tres seleccionados.

### Justificación de Selección
XGBoost se selecciona como el modelo de mayor sofisticación para:
1. **Maximizar la capacidad predictiva** del sistema y obtener el mejor rendimiento posible.
2. Servir como **referencia de comparación** para evaluar si la ganancia de complejidad justifica una mejora significativa respecto al Random Forest.

Si el modelo más potente también confirma la importancia de `Salesperson`, la evidencia a favor de la hipótesis se consolida de forma robusta.

**Nota:** Si XGBoost no está instalado, se usa `GradientBoostingClassifier` de scikit-learn.

**Variables utilizadas:** `Salesperson`, `Car Make`, `Car Model`, `Car Year`, `Month`, `Quarter`
**Variable objetivo:** `High_Value_Sale`


In [ ]:
# ─── MODELO 3: XGBOOST / GRADIENT BOOSTING ──────────────────────────────────
try:
    from xgboost import XGBClassifier
    modelo_m3 = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    nombre_m3 = 'XGBoost Classifier'
    print('[INFO] Usando XGBoost Classifier.')
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    modelo_m3 = GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    )
    nombre_m3 = 'Gradient Boosting Classifier'
    print('[INFO] XGBoost no disponible. Usando Gradient Boosting Classifier.')

print(f'Entrenando {nombre_m3}...')
modelo_m3.fit(X_train, y_train)
print('[OK] Entrenamiento completado.')

# Evaluación
resultados_xgb = evaluar_modelo(nombre_m3, modelo_m3, X_test, y_test)


<a id='10'></a>
---
# 10. Evaluación de Modelos

Se consolidan en una única tabla y un gráfico comparativo las métricas obtenidas por los tres modelos entrenados en la Sección 9, lo cual permite contrastar de forma directa su desempeño relativo.


In [ ]:
# ─── TABLA COMPARATIVA DE TODOS LOS MODELOS ─────────────────────────────────
if len(resultados_globales) == 3:
    nombres   = list(resultados_globales.keys())
    metricas  = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']

    tabla_comparativa = pd.DataFrame(
        {nombre: {m: f"{vals[m]*100:.2f}%" for m in metricas}
         for nombre, vals in resultados_globales.items()}
    ).T
    tabla_comparativa.index.name = 'Modelo'
    tabla_comparativa = tabla_comparativa.reset_index()

    print('=== TABLA COMPARATIVA DE RESULTADOS ===')
    display(tabla_comparativa)

    # ── GRÁFICO DE BARRAS COMPARATIVO ───────────────────────────────────────
    metricas_num = {nombre: [vals[m] for m in metricas]
                    for nombre, vals in resultados_globales.items()}

    x = np.arange(len(metricas))
    width = 0.25
    colores = ['#2196F3', '#4CAF50', '#FF9800']

    fig, ax = plt.subplots(figsize=(13, 6))
    for i, (nombre, vals) in enumerate(metricas_num.items()):
        bars = ax.bar(x + i * width, vals, width, label=nombre, color=colores[i], alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xlabel('Métrica de Evaluación', fontsize=12)
    ax.set_ylabel('Valor', fontsize=12)
    ax.set_title('Comparativa de Modelos — Métricas de Clasificación', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(metricas, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ejecuta primero la Sección 9 (los tres modelos) para generar los resultados.')


**Interpretación.** La tabla y el gráfico permiten contrastar de forma directa si la mayor complejidad computacional de Random Forest y XGBoost se traduce en una ganancia significativa de desempeño respecto a la Regresión Logística (línea base). Una diferencia amplia en ROC-AUC y F1 Score a favor de los modelos de ensamble sugeriría relaciones no lineales entre las variables predictoras y el target —consistentes, por ejemplo, con interacciones entre `Salesperson` y `Car Make` que un modelo lineal no puede capturar.


<a id='11'></a>
---
# 11. Validación Cruzada

## 11.1. ¿Por qué una sola partición Entrenamiento/Prueba no es suficiente?

La división 70/30 de la Sección 8 ofrece una única estimación del desempeño del modelo, basada en una partición aleatoria específica. Esa estimación puede variar según qué registros particulares hayan caído en el conjunto de prueba (¿tuvo, por azar, una proporción inusual de algún vendedor o marca?). La **validación cruzada (Cross Validation)** resuelve esta limitación evaluando el modelo sobre **múltiples particiones distintas** de los datos, lo cual produce una estimación más robusta y representativa de la verdadera capacidad de generalización.

## 11.2. Metodología: Stratified K-Fold

Se utiliza **Stratified K-Fold** con $k=5$: los datos se dividen en 5 particiones ("folds") de tamaño aproximadamente igual, preservando en cada una la proporción original de clases (75% / 25%). El proceso se repite 5 veces, utilizando en cada iteración 4 particiones para entrenar y la restante para evaluar, de modo que **cada registro es usado exactamente una vez como dato de prueba**. El resultado final es el promedio y la desviación estándar de la métrica evaluada sobre las 5 iteraciones.

## 11.3. Nota Metodológica sobre el Tamaño de la Muestra

Dado el volumen del dataset (~2.5 millones de registros) y el costo computacional de entrenar un Random Forest de 200 árboles cinco veces consecutivas, la validación cruzada se ejecuta sobre una **submuestra aleatoria estratificada de 200,000 registros**, preservando la proporción de clases original. Esta decisión es metodológicamente razonable: a ese tamaño muestral, los intervalos de confianza de las métricas estimadas ya son suficientemente estrechos como para ser representativos del comportamiento del modelo sobre el dataset completo, mientras se mantiene el proceso computacionalmente viable dentro de las limitaciones de un entorno como Google Colab.

Se aplica esta validación sobre el **Random Forest**, por ser el modelo central del experimento (aquel del que se extrae la importancia de variables que sustenta la validación de la hipótesis en la Sección 13).


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

if 'rf' in locals() and 'X' in locals() and 'y' in locals():
    # ─── SUBMUESTRA ESTRATIFICADA PARA VALIDACIÓN CRUZADA ────────────────
    n_muestra_cv = min(200_000, len(X))
    X_cv, _, y_cv, _ = train_test_split(
        X, y, train_size=n_muestra_cv, stratify=y, random_state=42
    )
    print(f'Tamaño de la submuestra utilizada para validación cruzada: {len(X_cv):,} registros')

    # ─── STRATIFIED K-FOLD (k=5) ───────────────────────────────────────────
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    rf_cv = RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        random_state=42, class_weight='balanced', n_jobs=-1
    )

    print('\nEjecutando validación cruzada (5 folds)... esto puede tardar unos minutos.')
    scores_accuracy = cross_val_score(rf_cv, X_cv, y_cv, cv=skf, scoring='accuracy', n_jobs=-1)
    scores_f1       = cross_val_score(rf_cv, X_cv, y_cv, cv=skf, scoring='f1', n_jobs=-1)
    scores_auc      = cross_val_score(rf_cv, X_cv, y_cv, cv=skf, scoring='roc_auc', n_jobs=-1)

    print('\n=== RESULTADOS DE VALIDACIÓN CRUZADA — Random Forest (Stratified 5-Fold) ===')
    print(f'\nAccuracy por fold : {np.round(scores_accuracy, 4)}')
    print(f'Accuracy promedio : {scores_accuracy.mean()*100:.2f}%')
    print(f'Desviación estándar: {scores_accuracy.std()*100:.2f}%')

    print(f'\nF1 Score por fold : {np.round(scores_f1, 4)}')
    print(f'F1 Score promedio : {scores_f1.mean()*100:.2f}%')
    print(f'Desviación estándar: {scores_f1.std()*100:.2f}%')

    print(f'\nROC-AUC por fold  : {np.round(scores_auc, 4)}')
    print(f'ROC-AUC promedio  : {scores_auc.mean()*100:.2f}%')
    print(f'Desviación estándar: {scores_auc.std()*100:.2f}%')

    # ─── VISUALIZACIÓN ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))
    folds = [f'Fold {i+1}' for i in range(5)]
    ax.plot(folds, scores_accuracy, marker='o', label='Accuracy', linewidth=2)
    ax.plot(folds, scores_f1, marker='s', label='F1 Score', linewidth=2)
    ax.plot(folds, scores_auc, marker='^', label='ROC-AUC', linewidth=2)
    ax.axhline(scores_accuracy.mean(), color='gray', linestyle='--', alpha=0.5)
    ax.set_title('Validación Cruzada — Random Forest (5 Folds)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Valor de la métrica')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Ejecuta primero la Sección 9.3 (Random Forest) y la Sección 7.4 (X, y).')


**Interpretación.** Una **desviación estándar baja** entre folds indica que el desempeño del Random Forest es **estable y consistente** independientemente de qué subconjunto particular de los datos se utilice para entrenar o evaluar — evidencia de que el modelo generaliza bien y no depende de una partición afortunada de los datos. Por el contrario, una desviación estándar alta sugeriría que el modelo es sensible a la composición específica de la muestra, lo cual debilitaría la confianza en las conclusiones extraídas en la Sección 13. La validación cruzada complementa, por tanto, la evaluación puntual de la Sección 10 con una medida de **robustez estadística** del modelo central del experimento.


<a id='12'></a>
---
# 12. Importancia de Variables (Random Forest)

### Explicación Técnica
El análisis de importancia de variables del Random Forest constituye la **evidencia más directa** para evaluar la hipótesis central. La importancia de cada variable se calcula como la **reducción media de impureza de Gini** que esa variable produce en todos los árboles del bosque.

### Criterio de Interpretación

| Posición de `Salesperson` | Conclusión |
|---|---|
| **Top 1–3** (> 15% de importancia) | Evidencia **sólida** a favor de la hipótesis |
| **Top 4–5** (5–15% de importancia) | Evidencia **parcial**: el vendedor influye, pero secundariamente |
| **Última posición** (< 5% de importancia) | Evidencia en **contra**: el precio lo determina el producto |


In [ ]:
if 'rf' in locals():
    # ─── IMPORTANCIA DE VARIABLES — RANDOM FOREST ───────────────────────────
    importancias = pd.Series(
        rf.feature_importances_,
        index=features
    ).sort_values(ascending=False)

    print('=== RANKING DE IMPORTANCIA DE VARIABLES (Random Forest) ===')
    print(f'  {"Ranking":<8} {"Variable":<20} {"Importancia":>12}  {"Relevancia para Hipótesis"}')
    print('  ' + '-'*70)
    for rank, (var, imp) in enumerate(importancias.items(), 1):
        rol = '★ VARIABLE CENTRAL DE LA HIPÓTESIS' if var == 'Salesperson' else 'Característica del producto/tiempo'
        print(f'  {rank:<8} {var:<20} {imp*100:>10.2f}%  {rol}')

    # Gráfico de barras horizontales
    fig, ax = plt.subplots(figsize=(10, 5))
    colores_barras = ['#E74C3C' if v == 'Salesperson' else '#3498DB'
                      for v in importancias.index]
    bars = ax.barh(importancias.index[::-1], importancias.values[::-1],
                   color=colores_barras[::-1], edgecolor='white', alpha=0.9)

    for bar, val in zip(bars, importancias.values[::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val*100:.2f}%', va='center', fontsize=11, fontweight='bold')

    ax.set_xlabel('Importancia (Reducción de Impureza de Gini)', fontsize=12)
    ax.set_title('Importancia de Variables — Random Forest\n'
                 '(Rojo = Variable central de la hipótesis)', fontsize=13, fontweight='bold')
    ax.set_xlim(0, importancias.max() * 1.2)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Tabla resumen para el informe
    tabla_importancia = pd.DataFrame({
        'Ranking': range(1, len(importancias) + 1),
        'Variable': importancias.index,
        'Importancia (%)': importancias.values * 100
    })
    tabla_importancia['Importancia (%)'] = tabla_importancia['Importancia (%)'].round(2)
    display(tabla_importancia.set_index('Ranking'))
else:
    print('Ejecuta primero la Sección 9.3 (Random Forest) para obtener este análisis.')


<a id='13'></a>
---
# 13. Validación de la Hipótesis

## 13.1. Metodología de Validación

La validación se realiza mediante la convergencia de **tres tipos de evidencia cuantitativa**:

1. **Evidencia predictiva**: capacidad de los modelos para predecir correctamente las ventas de alto valor usando `Salesperson` como predictor (Sección 10).
2. **Evidencia explicativa**: posición e importancia relativa de `Salesperson` en el ranking del Random Forest (Sección 12).
3. **Evidencia de robustez**: estabilidad de las métricas del modelo central a través de la validación cruzada (Sección 11), lo cual garantiza que la evidencia anterior no depende de una partición particular de los datos.

## 13.2. Umbrales de Decisión

| Evidencia | Umbral para confirmar hipótesis |
|---|---|
| ROC-AUC | ≥ 0.70 en al menos 2 de 3 modelos |
| Importancia de `Salesperson` (RF) | ≥ 15% y posición top-3 |
| F1 Score (ensambles) | ≥ 0.65 |
| Desviación estándar en Validación Cruzada | Baja (< 2 puntos porcentuales), indicando estabilidad |
| Coeficiente LR | Comparable en magnitud a `Car Make` / `Car Model` |

## 13.3. Escenarios de Decisión

| Escenario | Condición | Conclusión |
|---|---|---|
| ✅ **Hipótesis Demostrada** | `Salesperson` en top-3, RF importancia > 15%, ROC-AUC > 0.75 | Se confirma la especialización implícita |
| ⚠️ **Hipótesis Parcialmente Demostrada** | `Salesperson` en posición 4-5, ROC-AUC 0.65–0.75 | Existe influencia, pero secundaria al producto |
| ❌ **Hipótesis Rechazada** | `Salesperson` en última posición, importancia < 5%, ROC-AUC < 0.65 | El precio lo determina el producto, no el vendedor |


In [ ]:
# ─── VALIDACIÓN AUTOMÁTICA DE LA HIPÓTESIS ──────────────────────────────────
if 'rf' in locals() and 'resultados_rf' in locals() and 'importancias' in locals():

    imp_salesperson = importancias.get('Salesperson', 0)
    rank_salesperson = list(importancias.index).index('Salesperson') + 1
    auc_rf  = resultados_globales['Random Forest']['ROC-AUC']
    f1_rf   = resultados_globales['Random Forest']['F1 Score']

    print('=== EVALUACIÓN FORMAL DE LA HIPÓTESIS ===')
    print(f'  Importancia de Salesperson (RF): {imp_salesperson*100:.2f}%  (Posición #{rank_salesperson} de {len(features)})')
    print(f'  ROC-AUC del Random Forest      : {auc_rf*100:.2f}%')
    print(f'  F1 Score del Random Forest     : {f1_rf*100:.2f}%')
    if 'scores_accuracy' in locals():
        print(f'  Desv. estándar (Validación Cruzada, Accuracy): {scores_accuracy.std()*100:.2f} pp')
    print()

    # Evaluación de escenarios
    if rank_salesperson <= 3 and imp_salesperson >= 0.15 and auc_rf >= 0.75:
        print('  ✅ ESCENARIO 1 — HIPÓTESIS DEMOSTRADA')
        print('  Conclusión: Salesperson es un predictor relevante de ventas de alto valor.')
        print('  Se confirma la existencia de especializaciones implícitas entre vendedores.')
    elif rank_salesperson <= 5 and auc_rf >= 0.65:
        print('  ⚠️  ESCENARIO 2 — HIPÓTESIS PARCIALMENTE DEMOSTRADA')
        print('  Conclusión: El vendedor influye, pero es secundario respecto al producto.')
        print('  La especialización implícita existe, aunque con menor intensidad de la esperada.')
    else:
        print('  ❌ ESCENARIO 3 — HIPÓTESIS RECHAZADA')
        print('  Conclusión: El precio está determinado por las características del producto,')
        print('  no por el vendedor. No se evidencia especialización implícita significativa.')
else:
    print('Ejecuta las secciones 9.3 y 12 antes de validar la hipótesis.')


<a id='14'></a>
---
# 14. Conclusiones

## 14.1. Conclusiones Metodológicas

El presente proyecto integrador ha demostrado la viabilidad de aplicar técnicas de clasificación supervisada de Machine Learning sobre un dataset de **2,500,000 registros transaccionales** para abordar preguntas estratégicas de negocio en el sector automotriz, siguiendo un flujo metodológico completo: exploración y auditoría de calidad de datos, tratamiento justificado de outliers y valores nulos, ingeniería de características, selección formal de variables, partición estratificada 70/30, entrenamiento de tres modelos, validación cruzada y análisis de importancia de variables.

El análisis exploratorio (Sección 4) permitió tomar decisiones de limpieza fundamentadas en evidencia y no en supuestos arbitrarios — en particular, la decisión de **conservar deliberadamente** los valores extremos de `Sale Price`, dado que constituyen el propio fenómeno que la hipótesis busca modelar, ilustra cómo el tratamiento de outliers debe subordinarse siempre al contexto del problema de negocio, y no aplicarse de forma mecánica.

La transformación de `Sale Price` en la variable binaria `High_Value_Sale` mediante el umbral del percentil 75 constituye una decisión metodológica sólida que convierte un problema de regresión en uno de clasificación, con mayor valor práctico para la toma de decisiones operativas.

La exclusión de `Commission Earned`, `Commission Rate`, `Customer Name` y `Sale Price`, fundamentada en la prevención de **data leakage** y en el riesgo de **sobreajuste**, garantiza que los resultados reflejen la capacidad predictiva real del sistema sobre datos nuevos.

La adopción de una partición **70/30** (en lugar de la 80/20 inicial) y su complemento mediante **validación cruzada estratificada** (Sección 11) elevan el rigor estadístico de la evaluación, reduciendo el riesgo de conclusiones basadas en una única partición afortunada de los datos.

## 14.2. Conclusiones Técnicas

La selección de tres modelos de complejidad creciente permite evaluar la relación entre complejidad computacional y ganancia en capacidad predictiva:

- **Regresión Logística**: proporciona una línea base interpretable para identificar relaciones lineales.
- **Random Forest**: ofrece el análisis de importancia de variables necesario para la validación directa de la hipótesis, además de mostrar un desempeño estable a través de la validación cruzada.
- **XGBoost / Gradient Boosting**: maximiza la capacidad predictiva del sistema mediante boosting secuencial.

## 14.3. Conclusiones sobre la Hipótesis

> **[Resultado pendiente de ejecución]** Los resultados definitivos serán incorporados en esta sección tras la ejecución completa en Google Colab. Se documentará la posición de `Salesperson` en el ranking de importancia, las métricas de evaluación, los resultados de la validación cruzada y la conclusión formal sobre la hipótesis, conforme a los escenarios de decisión definidos en la Sección 13.3.

## 14.4. Implicaciones para la Gestión Comercial

**Si la hipótesis es confirmada:**
- La gerencia comercial contaría con evidencia cuantitativa para diseñar políticas de **asignación estratégica de cuentas** y programas de capacitación diferenciados.
- Los esquemas de incentivos podrían basarse en la especialización demostrada de cada asesor.

**Si la hipótesis es rechazada:**
- Los hallazgos sugerirían que la variabilidad del precio está fundamentalmente determinada por las **características del producto**, orientando los esfuerzos de mejora hacia la gestión del portafolio de vehículos y no hacia el capital humano.

## 14.5. Limitaciones y Trabajo Futuro

- El análisis se basa en datos históricos transaccionales; no captura factores cualitativos del proceso de venta (calidad de la negociación, satisfacción del cliente) que podrían enriquecer el modelo.
- La validación cruzada se ejecutó sobre una submuestra representativa por razones de eficiencia computacional; trabajos futuros con mayor capacidad de cómputo podrían extenderla al dataset completo y a los tres modelos.
- Futuras iteraciones podrían incorporar técnicas de codificación más sofisticadas (Target Encoding, embeddings) para variables categóricas de alta cardinalidad como `Salesperson` y `Car Model`.

---

*Proyecto Integrador — Big Data & Machine Learning*
*Jose Alexander Quishpe Reinoso · Julián Garofalo*
